In [10]:
import metapredict as mp
import pandas as pd
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
import statistics
from Bio.SeqUtils import ProtParam

In [11]:
tf = pd.read_csv('Full-DBDs.csv').dropna()
tf.reset_index(inplace=True)

In [12]:
len(tf)

1264

In [13]:

#Individual hydropathy score dictionary
hp ={'A':1.8, 'R':-4.5, 'N':-3.5, 'D':-3.5, 'C': 2.5, 'Q':-3.5, 'E':-3.5,
    'G':-0.4, 'H':-3.2, 'I': 4.5, 'L': 3.8, 'K':-3.9, 'M': 1.9, 'F': 2.8,
    'P':-1.6, 'S':-0.8, 'T':-0.7, 'W':-0.9, 'Y':-1.3, 'V': 4.2}

n_hp = {key: (value - min(hp.values())) / (max(hp.values()) - min(hp.values())) for key, value in hp.items()}

def fractions(sequence):
    prot_param = ProtParam.ProteinAnalysis(sequence)
    aa_counts = prot_param.count_amino_acids()
    total_aa_count = sum(aa_counts.values())
    aa_fractions = {aa: count / total_aa_count for aa, count in aa_counts.items()}

    return aa_fractions

def n_hp_score(sequence):
    prot_param = ProtParam.ProteinAnalysis(sequence)
    aa_counts = prot_param.count_amino_acids()
    total_aa_count = sum(aa_counts.values())
    aa_fractions = {aa: count / total_aa_count for aa, count in aa_counts.items()}
    hpscore = {amino_acid: n_hp[amino_acid] * aa_fractions[amino_acid] for amino_acid in n_hp}
    hpscore = sum(hpscore.values())
    return hpscore

def pondr_score(sequence):
    # Create a SeqRecord from the sequence in the DataFrame
    seq_record = SeqRecord(Seq(sequence), id="1")
    
    # Calculate the PONDR score for each sequence
    score = mp.predict_disorder(seq_record.seq)
    average = statistics.mean(score)
    return average

def calculate_net_charge(sequence):
    prot_param = ProtParam.ProteinAnalysis(sequence)
    net_charge = prot_param.charge_at_pH(7.4)  # Assuming physiological pH is 7.4
    return net_charge

def hp_score(sequence):
    prot_param = ProtParam.ProteinAnalysis(sequence)
    hp = prot_param.gravy()  # Assuming physiological pH is 7.4
    return hp

#tf['NetCharge'] = tf['BP Sequence'].apply(calculate_net_charge)

#tf['HP Score'] = tf['BP Sequence'].apply(hp_score)
#print(tf['Full-DBDs'])
tf['Fractions'] = tf['Full-DBDs'].apply(fractions)
columns_order = ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']
df_aa_fractions = pd.DataFrame(tf['Fractions'].apply(lambda x: [x[aa] for aa in columns_order]).tolist(), columns=columns_order)

tf['Normalized HP Score'] = tf['Full-DBDs'].apply(n_hp_score)
tf['PONDR Score'] = tf['Full-DBDs'].apply(pondr_score)

new = pd.DataFrame({
    'Protein': tf['Protein'],
    'UniProtID': tf['UniProt ID'],
    'Label': tf['Label'],
    'Effector domain Sequence': tf['Full-DBDs'],
    'Normalized HP Score' : tf['Normalized HP Score'],
    'PONDR Score' : tf['PONDR Score']
})

result = pd.concat([new, df_aa_fractions], axis=1)

result['f+']= result['R'] + result['K']
result['f-']= result['D'] + result['E']
result['|f+ - f-|']= abs(result['f+'] - result['f-'])

result.to_csv('Effector_domain_properties.csv',index=False)
print(len(result))

1264


In [14]:
result

,Protein,UniProtID,Label,Effector domain Sequence,Normalized HP Score,PONDR Score,A,C,D,E,...,Q,R,S,T,V,W,Y,f+,f-,|f+ - f-|
0,ANKZ1_HUMAN,Q9H8Y5,1.0,MSPAPDAAPAPASISLFDLSADAPVFQGLSLVSHAPGEALARAPRT...,0.417735,0.445286,0.113905,0.014793,0.048817,0.088757,...,0.062130,0.105030,0.073964,0.045858,0.041420,0.007396,0.016272,0.146450,0.137574,0.008876
1,ANM3_HUMAN,O60678,1.0,MCSLASGATGGRGAVENEEDLPELSDSGDEAAWEDEDDADLPHGKQ...,0.470347,0.214823,0.064182,0.018634,0.076605,0.078675,...,0.028986,0.026915,0.076605,0.057971,0.080745,0.010352,0.039337,0.107660,0.155280,0.047619
2,CHAP1_HUMAN,Q96JM3,1.0,MEAFQELRKPSARLECDHCSFRGTDYENVQIHMGTIHPEFCDEMDA...,0.403322,0.760843,0.065274,0.016971,0.036554,0.087467,...,0.036554,0.031332,0.143603,0.032637,0.040470,0.019582,0.011749,0.130548,0.124021,0.006527
3,DZIP1_HUMAN,Q86YF9,1.0,MQAEAADWFSSMPFQKHVYYPLASGPEGPDVAVAAAAAGAASMACA...,0.417528,0.515983,0.065934,0.012210,0.048840,0.113553,...,0.057387,0.040293,0.086691,0.046398,0.048840,0.009768,0.010989,0.136752,0.162393,0.025641
4,LMBL1_HUMAN,Q9Y468,1.0,MHLVAGDSPGSGPHLPATAFIIPASSATLGLPSSALDVSCFPREPI...,0.441460,0.436764,0.074468,0.026596,0.058511,0.075798,...,0.050532,0.035904,0.098404,0.047872,0.062500,0.023936,0.022606,0.078457,0.134309,0.055851
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1259,GTD2A_HUMAN,Q86UP8,45.0,MAQVAVSTLPVEEESSSETRMVVTFLVSALESMCKELAKSKAEVAC...,0.487971,0.149519,0.054482,0.022847,0.045694,0.080844,...,0.033392,0.038664,0.089631,0.056239,0.079086,0.010545,0.029877,0.110721,0.126538,0.015817
1260,GTD2B_HUMAN,Q6EKJ0,45.0,MAQVAVSTLPVEEESSSETRMVVTFLVSALESMCKELAKSKAEVAC...,0.489182,0.145745,0.052724,0.022847,0.045694,0.080844,...,0.031634,0.038664,0.087873,0.056239,0.080844,0.010545,0.029877,0.114236,0.126538,0.012302
1261,MBNL2_HUMAN,Q5VZF2,1.0,MALNVAPVRDTKPPTHLFMFPGTPLHPVPTFPVGPAIGTNTAISFA...,0.481439,0.559224,0.107955,0.045455,0.045455,0.039773,...,0.017045,0.056818,0.056818,0.107955,0.062500,0.000000,0.005682,0.096591,0.085227,0.011364
1262,COE2_HUMAN,Q9HAK2,42.0,MFGIQDTLGRGPTLKEKSLGAEMDSVRSWVRNVGVVDANVAAQSGV...,0.441720,0.512014,0.054496,0.008174,0.035422,0.035422,...,0.054496,0.059946,0.125341,0.054496,0.081744,0.002725,0.029973,0.100817,0.070845,0.029973


In [ ]:
counts_column1 = tf['Annotation Score'].value_counts()
counts_column2 = tf['Existence'].value_counts()

# Create histograms
plt.figure(figsize=(10, 6))

plt.subplot(1, 2, 1)
plt.bar(counts_column1.index, counts_column1.values)
plt.grid()
plt.title('Annotation Score')

plt.subplot(1, 2, 2)
plt.bar(counts_column2.index, counts_column2.values)
plt.title('Existence')
plt.xticks(rotation=90)
plt.grid()

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.hist(tf['NetCharge'], bins=50, edgecolor='black')
plt.title('Net Charge Distribution')

# Plot histogram for sequence length
tf['SequenceLength'] = tf['Sequence'].apply(len)
plt.subplot(1, 2, 2)
plt.hist(tf['SequenceLength'], bins=50, edgecolor='black')
plt.title('Sequence Length Distribution')

plt.tight_layout()
plt.show()

In [ ]:
unique_domains = tf['Domain'].unique().tolist()
unique_types = df['Type'].unique().tolist()

# Create a new DataFrame from the unique values
unique_df = pd.DataFrame({'Unique Domains': unique_domains, 'Unique Types': unique_types})

# Write the DataFrame to a CSV file
unique_df.to_csv('unique_values.csv', index=False)